In [1]:
import torch
from model.simple_nn import SimpleNN
import scanpy as sc
import numpy as np
import pandas as pd

In [2]:
dataset = sc.read_h5ad("combined_data.h5ad")
# dataset filter survival_3yr_label is NaN
dataset = dataset[dataset.obs['survival_3yr_label'].notna()]

In [3]:
import pandas as pd
import numpy as np
from typing import Optional, Union
import scipy.sparse as sp


def construct_feature_matrix(
    adata,
    expression_type: str = 'normalized',
    mutation_data: Optional[str] = None,
    include_survival_label: bool = False
) -> pd.DataFrame:
    """
    Construct a feature matrix from AnnData object combining gene expression and mutation data.
    
    Parameters
    ----------
    adata : AnnData
        The AnnData object containing gene expression and mutation data
    expression_type : str, default='normalized'
        Type of expression data to use:
        - 'normalized': Use the normalized expression matrix (adata.X)
        - 'raw': Use the raw counts matrix (adata.layers['counts'])
    mutation_data : str or None, default=None
        Which mutation data to include from adata.uns:
        - 'jiaming_data_colinear': Colinear mutation data
        - 'jiaming_data_full': Full mutation data
        - 'jiaming_data_intersect': Intersect mutation data
        - None: Don't include mutation data
    include_survival_label : bool, default=False
        Whether to include survival_3yr_label in the feature matrix
        
    Returns
    -------
    pd.DataFrame
        Feature matrix with samples as rows and features as columns.
        Column names will be prefixed:
        - 'gene_': for gene expression features
        - 'mut_': for mutation features
        - 'survival_3yr_label': if include_survival_label=True
    """
    
    # Get sample IDs
    sample_ids = adata.obs_names.tolist()
    
    # 1. Get gene expression data
    if expression_type == 'normalized':
        # Use normalized expression (X)
        expr_data = adata.X
    elif expression_type == 'raw':
        # Use raw counts
        if 'counts' not in adata.layers:
            raise ValueError("No 'counts' layer found in adata.layers")
        expr_data = adata.layers['counts']
    else:
        raise ValueError(f"expression_type must be 'normalized' or 'raw', got {expression_type}")
    
    # Convert sparse matrix to dense if necessary
    if hasattr(expr_data, 'toarray'):
        # It's a sparse matrix, convert to dense
        expr_data = expr_data.toarray()
    elif hasattr(expr_data, 'todense'):
        # Alternative sparse matrix format
        expr_data = np.asarray(expr_data.todense())
    
    # Ensure it's a 2D array
    if expr_data.ndim == 1:
        expr_data = expr_data.reshape(-1, 1)
    
    # Convert to DataFrame with gene names
    gene_names = adata.var_names.tolist()
    
    # Verify dimensions match
    if expr_data.shape[1] != len(gene_names):
        raise ValueError(f"Expression data has {expr_data.shape[1]} features but found {len(gene_names)} gene names")
    
    expr_df = pd.DataFrame(
        expr_data,
        index=sample_ids,
        columns=[f'gene_{gene}' for gene in gene_names]
    )
    
    # 2. Add mutation data if requested
    if mutation_data is not None:
        if mutation_data not in adata.uns:
            raise ValueError(f"'{mutation_data}' not found in adata.uns")
        
        mut_df = adata.uns[mutation_data]
        
        # Ensure mutation data is a DataFrame
        if not isinstance(mut_df, pd.DataFrame):
            # If it's an array, convert to DataFrame
            mut_df = pd.DataFrame(mut_df)
        
        # Prefix mutation column names
        mut_df.columns = [f'mut_{col}' for col in mut_df.columns]
        
        # Handle missing samples by creating a zero-filled DataFrame for all samples
        # then updating with available mutation data
        all_mut_df = pd.DataFrame(
            0,  # Fill with zeros
            index=sample_ids,
            columns=mut_df.columns
        )
        
        # Update with available mutation data
        # Find common samples between mutation data and sample_ids
        common_samples = [idx for idx in mut_df.index if idx in sample_ids]
        
        if len(common_samples) > 0:
            all_mut_df.loc[common_samples] = mut_df.loc[common_samples]
        
        # Combine expression and mutation data
        feature_matrix = pd.concat([expr_df, all_mut_df], axis=1)
    else:
        feature_matrix = expr_df
    
    # 3. Add survival label if requested
    if include_survival_label:
        if 'survival_3yr_label' in adata.obs.columns:
            feature_matrix['survival_3yr_label'] = adata.obs['survival_3yr_label'].values
        else:
            print("Warning: 'survival_3yr_label' not found in adata.obs")
    
    return feature_matrix


# Example usage function
def prepare_ml_data(
    adata,
    expression_type: str = 'normalized',
    mutation_data: Optional[str] = 'jiaming_data_full',
    test_size: float = 0.2,
    random_state: int = 42
):
    """
    Prepare data for machine learning by constructing features and splitting into train/test.
    
    Returns
    -------
    tuple
        (X_train, X_test, y_train, y_test, feature_names)
    """
    from sklearn.model_selection import train_test_split
    
    # Get feature matrix without survival label
    X_df = construct_feature_matrix(
        adata,
        expression_type=expression_type,
        mutation_data=mutation_data,
        include_survival_label=False
    )
    
    # Get survival labels
    if 'survival_3yr_label' not in adata.obs.columns:
        raise ValueError("'survival_3yr_label' not found in adata.obs")
    
    y = adata.obs['survival_3yr_label'].values
    
    # Convert to numpy arrays
    X = X_df.values
    feature_names = X_df.columns.tolist()
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    return X_train, X_test, y_train, y_test, feature_names


# Utility function to explore the data structure
def explore_anndata_structure(adata):
    """
    Print a summary of the AnnData object structure.
    """
    print(f"AnnData object shape: {adata.shape}")
    print(f"\nExpression data (X):")
    print(f"  Type: {type(adata.X)}")
    print(f"  Shape: {adata.X.shape}")
    if hasattr(adata.X, 'nnz'):
        print(f"  Sparse matrix with {adata.X.nnz} non-zero elements")
    
    if 'counts' in adata.layers:
        print(f"\nRaw counts layer:")
        print(f"  Type: {type(adata.layers['counts'])}")
        print(f"  Shape: {adata.layers['counts'].shape}")
        if hasattr(adata.layers['counts'], 'nnz'):
            print(f"  Sparse matrix with {adata.layers['counts'].nnz} non-zero elements")
    
    print(f"\nobs columns: {list(adata.obs.columns)}")
    print(f"var columns: {list(adata.var.columns)}")
    
    print("\nuns keys:")
    for key in adata.uns_keys():
        if key.startswith('jiaming_'):
            data = adata.uns[key]
            if hasattr(data, 'shape'):
                print(f"  {key}: shape {data.shape}")
            else:
                print(f"  {key}: type {type(data)}")
                if isinstance(data, pd.DataFrame):
                    print(f"    DataFrame shape: {data.shape}")
                    print(f"    Index length: {len(data.index)}")


# Debug function to test data extraction
def debug_expression_data(adata, expression_type='normalized'):
    """
    Debug function to examine expression data structure.
    """
    print(f"Debugging {expression_type} expression data:")
    
    if expression_type == 'normalized':
        expr_data = adata.X
    else:
        expr_data = adata.layers.get('counts', None)
        if expr_data is None:
            print("No 'counts' layer found!")
            return
    
    print(f"  Raw type: {type(expr_data)}")
    print(f"  Shape: {expr_data.shape if hasattr(expr_data, 'shape') else 'No shape attribute'}")
    
    # Try to get a small sample
    try:
        if hasattr(expr_data, 'toarray'):
            sample = expr_data[:5, :5].toarray()
        else:
            sample = expr_data[:5, :5]
        print(f"  Sample (5x5):\n{sample}")
    except Exception as e:
        print(f"  Could not extract sample: {e}")
    
    return expr_data

def count_feature_types(feature_matrix_or_names):
    """
    Count the number of DNA (mutation) and RNA (gene expression) features.
    
    Parameters
    ----------
    feature_matrix_or_names : pd.DataFrame or list
        Either a feature matrix DataFrame or a list of feature names
        
    Returns
    -------
    dict
        Dictionary with counts:
        - 'n_rna_features': Number of gene expression features
        - 'n_dna_features': Number of mutation features  
        - 'n_other_features': Number of other features (e.g., survival labels)
        - 'total_features': Total number of features
        - 'rna_features': List of RNA feature names
        - 'dna_features': List of DNA feature names
        - 'other_features': List of other feature names
    """
    # Get feature names
    if isinstance(feature_matrix_or_names, pd.DataFrame):
        feature_names = feature_matrix_or_names.columns.tolist()
    else:
        feature_names = list(feature_matrix_or_names)
    
    # Count different feature types
    rna_features = [f for f in feature_names if f.startswith('gene_')]
    dna_features = [f for f in feature_names if f.startswith('mut_')]
    other_features = [f for f in feature_names if not f.startswith('gene_') and not f.startswith('mut_')]
    
    counts = {
        'n_rna_features': len(rna_features),
        'n_dna_features': len(dna_features),
        'n_other_features': len(other_features),
        'total_features': len(feature_names),
        'rna_features': rna_features,
        'dna_features': dna_features,
        'other_features': other_features
    }
    
    return counts

In [4]:
# First, explore the structure to understand your data
anndata = dataset
print("=== Exploring AnnData Structure ===")
explore_anndata_structure(dataset)

# Debug expression data if needed
print("\n=== Debugging Expression Data ===")
debug_expression_data(anndata, 'normalized')

# 1. Create feature matrix with normalized expression and full mutation data
print("\n=== Creating Feature Matrices ===")
features_norm_full = construct_feature_matrix(
    anndata,
    expression_type='normalized',
    mutation_data='jiaming_data_full'
)
print(f"Feature matrix shape: {features_norm_full.shape}")
print(f"Feature matrix columns sample: {list(features_norm_full.columns[:5])}")

# 2. Create feature matrix with raw counts and colinear mutation data
features_raw_colinear = construct_feature_matrix(
    anndata,
    expression_type='raw',
    mutation_data='jiaming_data_colinear'
)

# 3. Create feature matrix with only gene expression (no mutations)
features_expr_only = construct_feature_matrix(
    anndata,
    expression_type='normalized',
    mutation_data=None
)

# 4. Prepare data for ML with train/test split
X_train, X_test, y_train, y_test, feature_names = prepare_ml_data(
    anndata,
    expression_type='normalized',
    mutation_data='jiaming_data_full',
    test_size=0.2
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"Feature distribution: {sum(1 for f in feature_names if f.startswith('gene_'))} genes, "
        f"{sum(1 for f in feature_names if f.startswith('mut_'))} mutations")

=== Exploring AnnData Structure ===
AnnData object shape: (246, 914)

Expression data (X):
  Type: <class 'anndata._core.views.SparseCSCView'>
  Shape: (246, 914)
  Sparse matrix with 206824 non-zero elements

Raw counts layer:
  Type: <class 'anndata._core.views.SparseCSCView'>
  Shape: (246, 914)
  Sparse matrix with 206824 non-zero elements

obs columns: ['survival_3yr_label']
var columns: ['vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable', 'KM_markers', 'jm_full_genes', 'jm_colinear_genes', 'jm_intersect_genes']

uns keys:
  jiaming_data_colinear: shape (504, 145)
  jiaming_data_full: shape (504, 401)
  jiaming_data_intersect: shape (465, 18)

=== Debugging Expression Data ===
Debugging normalized expression data:
  Raw type: <class 'anndata._core.views.SparseCSCView'>
  Shape: (246, 914)
  Sample (5x5):
[[1.2271996e-01 2.1860221e-02 1.9103946e-01 6.7393279e-01 0.0000000e+00]
 [9.0700674e-01 1.2770356e-02 7.7960737e-02 3.5644954e-01 0.

In [5]:
feature_matrix = construct_feature_matrix(
    anndata,
    expression_type='normalized',
    mutation_data='jiaming_data_full'
)

# Get labels
labels = anndata.obs['survival_3yr_label'].values

# Count features
feature_counts = count_feature_types(feature_matrix)
print(f"Total features: {feature_counts['total_features']}")
print(f"RNA features: {feature_counts['n_rna_features']}")
print(f"DNA features: {feature_counts['n_dna_features']}")


Total features: 1315
RNA features: 914
DNA features: 401


In [6]:
from model.simple_nn import SimpleNN, k_fold_cross_validation, plot_cv_results

def train_with_simple_nn():
    """Train using the original SimpleNN with embedding layers."""    
    # Model configuration
    model_kwargs = {
        'embedding_dim': 10
        # dna_features and rna_features will be added automatically
    }
    
    # Run k-fold cross-validation
    cv_results = k_fold_cross_validation(
        feature_matrix=feature_matrix,
        labels=labels,
        model_class=SimpleNN,
        model_kwargs=model_kwargs,
        k=5,
        n_epochs=50,
        batch_size=32,
        learning_rate=0.001
    )
    
    # Plot results
    plot_cv_results(cv_results)
    
    # Get average performance
    avg_metrics = cv_results['average_metrics']
    print(f"\nAverage 5-Fold CV Performance:")
    print(f"Accuracy: {avg_metrics['avg_accuracy']:.3f} ± {avg_metrics['std_accuracy']:.3f}")
    print(f"AUC: {avg_metrics['avg_auc']:.3f} ± {avg_metrics['std_auc']:.3f}")
    
    return cv_results